# 01 — Data Quality and Filtering

This notebook begins the assessment of the **ASSISTments 2009–2010 Skill Builder** data. The aim is to understand the dataset before feature engineering or modeling and to document decisions that affect the validity of later student-proficiency predictions.

## Objectives

- Confirm the dataset's shape, schema, types, and identifiers.
- Measure missingness, duplicates, invalid values, and suspicious outliers.
- Examine the target (`correct`) and important student, skill, problem, and class dimensions.
- Explore longitudinal coverage and ordering using `order_id`.
- Identify leakage risks and record data-cleaning decisions for downstream modeling.

> **Responsible-use note:** These interaction records describe performance within one learning platform. They are not measures of intelligence, motivation, disability, or general ability, and any findings should be interpreted as teacher-facing decision support rather than automatic educational decisions.

## Imports and notebook setup

In [55]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

Python: 3.13.14
pandas: 3.0.5
NumPy:  2.5.2


## Project paths

The path setup below works when VS Code starts the notebook from either the repository root or the `notebooks/` directory. The corrected collapsed file is the primary analysis dataset because it consolidates multi-skill student–problem records.

In [56]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "skill_builder_data_corrected_collapsed.csv"
DATA_DICTIONARY_PATH = (
    DATA_DIR
    / "data_dictionary"
    / "assistments_2009_2010_skill_builder_data_dictionary.csv"
)

assert RAW_DATA_PATH.is_file(), f"Raw dataset not found: {RAW_DATA_PATH}"
assert DATA_DICTIONARY_PATH.is_file(), f"Data dictionary not found: {DATA_DICTIONARY_PATH}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data:     {RAW_DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Dictionary:   {DATA_DICTIONARY_PATH.relative_to(PROJECT_ROOT)}")

Project root: W:\Workstation ExtDrive\007 Data Science\003 Data Science Projects\2026_p019 assistments_2009_2010
Raw data:     data\raw\skill_builder_data_corrected_collapsed.csv
Dictionary:   data\data_dictionary\assistments_2009_2010_skill_builder_data_dictionary.csv


## Next: load and inspect the data

Suggested next steps are to load the data with explicit types for identifier/text fields, compare the observed schema with the data dictionary, and build a compact quality summary before beginning univariate or longitudinal exploration.

The data also needs to be converted to UTF-8. Exclude the answer_text column which contains non-utf characters

In [57]:
# create path for saving utf-8 encoded data without answer_text column
UTF8_DATA_PATH = (
    DATA_DIR / "processed" / "skill_builder_data_without_answer_text.csv"
)
UTF8_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    RAW_DATA_PATH,
    encoding="cp1252",
    usecols=lambda column: column != "answer_text",
    low_memory=False,
)

df.to_csv(
    UTF8_DATA_PATH,
    encoding="utf-8",
    index=False,
)

In [58]:
# load data again to ensure it's in utf-8 format and without the answer_text column
df = pd.read_csv(UTF8_DATA_PATH)

C:\Users\helsh\AppData\Local\Temp\ipykernel_6748\4259348844.py:2: DtypeWarning: Columns (0: skill_id, 1: skill_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(UTF8_DATA_PATH)


In [59]:
# data summary
# 1. View basic info: row count, column names, non-null counts, and data types
print("--- DataFrame Info ---")
df.info()

# 2. View statistical summary for numerical columns (count, mean, std, min, percentiles, max)
print("\n--- Numerical Summary ---")
print(df.describe())

# 3. View statistical summary for all columns (including text/categorical)
print("\n--- Full Summary (Including Categorical) ---")
print(df.describe(include="all"))

# 4. Check for missing values count per column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 5. Check the first 5 rows to see actual data values
print("\n--- First 5 Rows ---")
print(df.head())

--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 346860 entries, 0 to 346859
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   row_id                346860 non-null  int64  
 1   order_id              346860 non-null  int64  
 2   assignment_id         346860 non-null  int64  
 3   user_id               346860 non-null  int64  
 4   assistment_id         346860 non-null  int64  
 5   problem_id            346860 non-null  int64  
 6   original              346860 non-null  int64  
 7   correct               346860 non-null  int64  
 8   attempt_count         346860 non-null  int64  
 9   ms_first_response     346860 non-null  int64  
 10  tutor_mode            346860 non-null  str    
 11  answer_type           346860 non-null  str    
 12  sequence_id           346860 non-null  int64  
 13  student_class_id      346860 non-null  int64  
 14  position              346860 non-null  i

In [60]:
# data quality and filtering

# only keep original questions, not scaffolding questions. Therefore keep original = 1.
df = df[df["original"] == 1]

# ms_first_response < 0. Only a few rows of this, filter them out. (this also filters out overlap_time < 0)
df = df[df["ms_first_response"] >= 0]

# answer_type = "open_response" may be automatically counted as incorrect, not sure. Only a few rows, exclude
df = df[df["answer_type"] != "open_response"]

# skill_id is essential for feature engineering, exclude missing skill_id rows. skill_name will not be used for modeling, may be useful for EDA or reporting
df = df[df["skill_id"].notnull()]

# for bottom_hint possible values in data are 0, 1, and "NA". convert null to 0. make data type int.
df["bottom_hint"] = df["bottom_hint"].fillna(0)
df["bottom_hint"] = df["bottom_hint"].astype(int)

# drop columns: opportunity_original, type, answer_id
df = df.drop(columns=["opportunity_original", "type", "answer_id"])



In [61]:
# check data summary after filtering
print("\n--- DataFrame Info After Filtering ---")
df.info()


--- DataFrame Info After Filtering ---
<class 'pandas.DataFrame'>
Index: 259386 entries, 0 to 283104
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   row_id             259386 non-null  int64 
 1   order_id           259386 non-null  int64 
 2   assignment_id      259386 non-null  int64 
 3   user_id            259386 non-null  int64 
 4   assistment_id      259386 non-null  int64 
 5   problem_id         259386 non-null  int64 
 6   original           259386 non-null  int64 
 7   correct            259386 non-null  int64 
 8   attempt_count      259386 non-null  int64 
 9   ms_first_response  259386 non-null  int64 
 10  tutor_mode         259386 non-null  str   
 11  answer_type        259386 non-null  str   
 12  sequence_id        259386 non-null  int64 
 13  student_class_id   259386 non-null  int64 
 14  position           259386 non-null  int64 
 15  base_sequence_id   259386 non-null  int64 
 

In [62]:
# View statistical summary for all columns (including text/categorical) after filtering
print("\n--- Full Summary (Including Categorical) after filtering ---")
df.describe(include="all")


--- Full Summary (Including Categorical) after filtering ---


,row_id,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,answer_type,sequence_id,student_class_id,position,base_sequence_id,skill_id,skill_name,teacher_id,school_id,hint_count,hint_total,overlap_time,template_id,first_action,bottom_hint,opportunity
count,"259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000",259386,259386,"259,386.000","259,386.000","259,386.000","259,386.000",259386,251392,"259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000","259,386.000"
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,4,NaN,NaN,NaN,NaN,146,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tutor,algebra,NaN,NaN,NaN,NaN,47,Conversion of Fraction Decimals Percents,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,259127,218104,NaN,NaN,NaN,NaN,18736,18739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,"165,936.367","30,489,479.403","273,670.025","83,303.544","46,608.057","81,622.049",1.000,0.658,1.579,"50,337.005",NaN,NaN,"7,347.665","12,901.953",52.169,"6,794.355",NaN,NaN,"46,949.713","3,056.423",0.437,2.406,"62,495.630","39,248.742",0.118,0.096,14.912
std,"96,431.381","5,215,213.473","10,547.275","7,306.701","10,630.579","23,004.530",0.000,0.474,13.433,"373,596.283",NaN,NaN,"1,454.062",765.833,61.871,"1,178.483",NaN,NaN,"15,676.873","1,880.637",1.171,1.875,"401,096.977","11,956.412",0.390,0.294,69.555
min,1.000,"20,224,180.000","217,900.000",14.000,86.000,86.000,1.000,0.000,0.000,0.000,NaN,NaN,"5,870.000","11,644.000",1.000,"5,870.000",NaN,NaN,"11,158.000",1.000,0.000,0.000,"-7,241,868.000",86.000,0.000,0.000,1.000
25%,"86,525.250","26,486,304.500","266,709.000","78,968.000","37,856.000","60,141.000",1.000,0.000,1.000,"9,879.000",NaN,NaN,"6,039.000","12,352.000",9.000,"5,970.000",NaN,NaN,"42,999.000","2,770.000",0.000,0.000,"11,858.000","30,047.000",0.000,0.000,3.000
50%,"166,313.500","30,766,392.500","271,491.500","80,228.000","47,305.000","85,823.000",1.000,1.000,1.000,"21,665.000",NaN,NaN,"6,943.000","12,617.000",24.000,"6,409.000",NaN,NaN,"45,778.000","2,770.000",0.000,3.000,"26,254.500","31,001.000",0.000,0.000,6.000
75%,"250,158.750","34,599,987.750","278,459.750","87,992.000","51,402.000","90,629.000",1.000,1.000,1.000,"48,219.000",NaN,NaN,"8,116.000","13,241.000",81.000,"7,012.000",NaN,NaN,"58,597.000","5,056.000",0.000,4.000,"60,331.500","46,338.000",0.000,0.000,13.000


In [63]:
# 4. Check for missing values count per column after filtering
print("\n--- Missing Values Count after filtering ---")
df.isnull().sum()




--- Missing Values Count after filtering ---


row_id                  0
order_id                0
assignment_id           0
user_id                 0
assistment_id           0
problem_id              0
original                0
correct                 0
attempt_count           0
ms_first_response       0
tutor_mode              0
answer_type             0
sequence_id             0
student_class_id        0
position                0
base_sequence_id        0
skill_id                0
skill_name           7994
teacher_id              0
school_id               0
hint_count              0
hint_total              0
overlap_time            0
template_id             0
first_action            0
bottom_hint             0
opportunity             0
dtype: int64

In [64]:
# Check the first 5 rows to see actual data values after filtering
print("\n--- First 5 Rows after filtering ---")
df.head()


--- First 5 Rows after filtering ---


,row_id,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,answer_type,sequence_id,student_class_id,position,base_sequence_id,skill_id,skill_name,teacher_id,school_id,hint_count,hint_total,overlap_time,template_id,first_action,bottom_hint,opportunity
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,algebra,5948,13241,126,5948,1_13,Box and Whisker,22763,73,0,3,32454,30799,0,0,1
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,algebra,5948,13241,126,5948,1_13,Box and Whisker,22763,73,0,3,4922,30799,0,0,2
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,algebra,5948,11816,22,5948,1_13,Box and Whisker,22763,73,0,3,42000,30799,0,0,1
3,4,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,algebra,5948,11816,22,5948,1_13,Box and Whisker,22763,73,0,3,4859,30059,0,0,2
4,5,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,algebra,5948,11816,22,5948,1_15,Box and Whisker,22763,73,3,4,124564,30060,0,0,3


In [65]:
# Sort df by user_id, order_id. order_id is in chronological order, so this will sort the data by user_id and then by time.
df = df.sort_values(by=["user_id", "order_id"]).reset_index(drop=True)

In [66]:
# re-order columns to have user_id, order_id, skill_id, skill_name, and then the rest of the columns with 'correct' column being the last (as it is the target variable)
df = df[["user_id", "order_id", "skill_id", "skill_name"] + [col for col in df.columns if col not in ["user_id", "order_id", "skill_id", "skill_name"]]]
df = df[[col for col in df.columns if col != "correct"] + ["correct"]] # make correct column the last column in the dataframe



In [67]:
# Break up skill_id into separate columns with one-hot encoding for each skill_id

# Split composite skill IDs and one-hot encode individual skills
MAX_SKILLS = 200

skill_text = (
    df["skill_id"]
    .astype("string")
    .replace({
        "": pd.NA,
        "NA": pd.NA,
        "nan": pd.NA,
    })
    # Convert "312.0" to "312" and "2.0_37.0" to "2_37"
    .str.replace(r"\.0+(?=_|$)", "", regex=True)
)

# Count and validate the individual skill IDs
skill_tokens = skill_text.str.split("_").explode().dropna()

invalid_tokens = skill_tokens[
    ~skill_tokens.str.fullmatch(r"\d+")
].unique()

if len(invalid_tokens) > 0:
    raise ValueError(
        f"Non-integer skill IDs found: {invalid_tokens[:10].tolist()}"
    )

n_individual_skills = skill_tokens.nunique()
print(f"Unique individual skills: {n_individual_skills}")

if n_individual_skills > MAX_SKILLS:
    raise ValueError(
        f"{n_individual_skills} skills exceeds the "
        f"MAX_SKILLS threshold of {MAX_SKILLS}."
    )

# Create one-hot columns
skill_encoded = (
    skill_text
    .fillna("")
    .str.get_dummies(sep="_")
)

# Put columns in numeric skill-ID order
skill_encoded = skill_encoded.reindex(
    columns=sorted(skill_encoded.columns, key=int)
)

skill_encoded.columns = [
    f"Skill_{skill_id}"
    for skill_id in skill_encoded.columns
]

# uint8 only needs one byte per value
skill_encoded = skill_encoded.astype("uint8")

# Make the cell safe to rerun by removing previously created Skill_* columns
existing_skill_columns = [
    column for column in df.columns
    if column.startswith("Skill_")
]

df = pd.concat(
    [
        df.drop(columns=existing_skill_columns),
        skill_encoded,
    ],
    axis=1,
)

print(f"Created {skill_encoded.shape[1]} one-hot columns.")
print(
    f"Approximate added memory: "
    f"{skill_encoded.memory_usage(index=False).sum() / 1024**2:.1f} MB"
)

df.head()

Unique individual skills: 123
Created 123 one-hot columns.
Approximate added memory: 30.4 MB


,user_id,order_id,skill_id,skill_name,row_id,assignment_id,assistment_id,problem_id,original,attempt_count,ms_first_response,tutor_mode,answer_type,sequence_id,student_class_id,position,base_sequence_id,teacher_id,school_id,hint_count,hint_total,overlap_time,template_id,first_action,bottom_hint,opportunity,correct,Skill_1,Skill_2,Skill_4,Skill_5,Skill_8,Skill_9,Skill_10,Skill_11,Skill_12,Skill_13,Skill_14,Skill_15,Skill_16,Skill_17,Skill_18,Skill_21,Skill_22,Skill_24,Skill_25,Skill_26,Skill_27,Skill_32,Skill_34,Skill_35,Skill_37,Skill_39,Skill_40,Skill_42,Skill_43,Skill_46,Skill_47,Skill_48,Skill_49,Skill_50,Skill_51,Skill_53,Skill_54,Skill_58,Skill_61,Skill_63,Skill_64,Skill_65,Skill_67,Skill_69,Skill_70,Skill_74,Skill_75,Skill_76,Skill_77,Skill_79,Skill_80,Skill_81,Skill_82,Skill_83,Skill_84,Skill_85,Skill_86,Skill_91,Skill_92,Skill_94,Skill_96,Skill_97,Skill_99,Skill_101,Skill_102,Skill_104,Skill_105,Skill_110,Skill_163,Skill_165,Skill_166,Skill_173,Skill_190,Skill_193,Skill_203,Skill_204,Skill_217,Skill_221,Skill_276,Skill_277,Skill_278,Skill_279,Skill_280,Skill_290,Skill_292,Skill_293,Skill_294,Skill_295,Skill_296,Skill_297,Skill_298,Skill_299,Skill_301,Skill_303,Skill_307,Skill_308,Skill_309,Skill_310,Skill_311,Skill_312,Skill_314,Skill_317,Skill_321,Skill_322,Skill_323,Skill_324,Skill_325,Skill_331,Skill_333,Skill_334,Skill_340,Skill_343,Skill_346,Skill_348,Skill_350,Skill_356,Skill_362,Skill_365,Skill_367,Skill_368,Skill_371,Skill_375,Skill_378
0,14,21617623,2_37_70,Circle Graph,3958,263599,53412,93383,1,1,26271,tutor,algebra,7118,12495,1,7118,42972,1,2,2,41131,52570,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,14,21617632,2_37_70,Circle Graph,3959,263599,53436,93407,1,1,29123,tutor,algebra,7118,12495,1,7118,42972,1,0,2,29123,52570,0,0,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14,21617641,2_37_70,Circle Graph,3960,263599,53429,93400,1,1,13779,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19905,52570,1,1,3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,14,21617650,2_37_70,Circle Graph,3961,263599,53448,93419,1,1,16901,tutor,algebra,7118,12495,1,7118,42972,1,2,2,22600,52570,1,1,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,14,21617659,2_37_70,Circle Graph,3962,263599,53449,93420,1,1,11079,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19704,52570,1,1,5,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [68]:
# drop skill_id column since we have one-hot encoded skill columns now. skill_name will be kept for EDA and reporting purposes.
df = df.drop(columns=["skill_id"])

In [69]:
# export filtered and one-hot encoded data to csv for later use
PROCESSED_DATA_PATH = DATA_DIR / "processed" / "skill_builder_data_filtered_onehot.csv"
df.to_csv(
    PROCESSED_DATA_PATH,
    encoding="utf-8",
    index=False,
)

In [71]:
# how many unique user_ids are there in the filtered dataset?
unique_user_ids = df["user_id"].nunique()
unique_user_ids

4163